In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["MKL_SERVICE_FORCE_INTEL"]="1"
# os.environ["OMP_NUM_THREADS"] = "8"  # 控制MKL使用的线程数
# os.environ["MKL_NUM_THREADS"] = "8"
# os.environ["BLAS"] = "MKL"
# os.environ["USE_CUSOLVER"] = "1"

import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoFeatureExtractor, SwinForImageClassification,
    ViTImageProcessor, ViTForImageClassification,
    Trainer, TrainingArguments, EvalPrediction,
)
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

from meft import MeftConfig, MeftTrainer
import meft

In [12]:
import random
import numpy as np
from transformers import set_seed as hf_set_seed
seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
hf_set_seed(seed)  # transformers 内部用到的随机也统一
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [13]:
model_name = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    num_labels=100,
    ignore_mismatched_sizes=True,
    # device_map="auto",
    device_map="cuda:0",
)

dataset = load_dataset("cifar100")

def transform(examples):
    inputs = processor(
        images=[x.convert("RGB") for x in examples["img"]],
        return_tensors="pt"
    )
    return {
        "pixel_values": inputs.pixel_values,
        "label": examples["fine_label"]
    }

dataset = dataset.with_transform(transform)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 371,812 || all params: 86,247,368 || trainable%: 0.4311


In [15]:
def compute_metrics(eval_pred: EvalPrediction):
    evaluation = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return evaluation.compute(predictions=predictions, references=labels)

In [16]:
trainer = MeftTrainer[Trainer](
    model=model,
    args=TrainingArguments(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=32,
        num_train_epochs=1,
        learning_rate=1e-3,
        weight_decay=1e-2,
        warmup_ratio=0.1,    # 新加
        lr_scheduler_type="cosine",
        bf16=True,
        bf16_full_eval=True,
        # deepspeed={
        #     "train_batch_size": "auto",
        #     "gradient_accumulation_steps": "auto",
        #     "zero_optimization": {
        #         "stage": 1,
        #     },
        # },
        use_liger_kernel=True,
        logging_steps=10,
        report_to="none",
        remove_unused_columns=False,
        label_names=["labels"],
    ),
    data_collator=None,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    meft_config=MeftConfig(
        patch_locations=(
            "norm",
            # "attn_in",
            # "attn_out",
            # "mlp_in",
            # "mlp_out",
            # "act_fn",
            "ckpt_attn",
            "ckpt_mlp",
            # "ckpt_layer",
        ),
        compress_kwargs={
            "rank": 0.125,
            # "niter": 1,
        },
        # compress_workers=2,
    ),
)

Applying patch to vit model in: ('norm', 'ckpt_attn', 'ckpt_mlp')


/home/shijx/chenxy/LoRAct/meft/patch/models/vit.py:49: CheckpointViTMLPWarning: ViT only supports checkpointing the first layer of MLP.
  warnings.warn("ViT only supports checkpointing the first layer of MLP.", CheckpointViTMLPWarning)


In [ ]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


W1224 16:16:22.740000 920390 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] torch._dynamo hit config.recompile_limit (8)
W1224 16:16:22.740000 920390 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8]    function: 'inner' (/home/shijx/miniconda3/envs/loract/lib/python3.10/site-packages/torch/_dynamo/external_utils.py:66)
W1224 16:16:22.740000 920390 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8]    last reason: 0/7: GLOBAL_STATE changed: autocast 
W1224 16:16:22.740000 920390 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W1224 16:16:22.740000 920390 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html


Step,Training Loss
10,4.600200
20,4.523100
30,4.380400
40,4.112300
50,3.677400
60,3.159000
70,2.736900
80,2.312900
90,1.938400
100,1.613400


KeyboardInterrupt: 

: 

In [ ]:
trainer.evaluate()

{'eval_loss': 0.5159130096435547,
 'eval_accuracy': 0.8918,
 'eval_runtime': 38.747,
 'eval_samples_per_second': 258.085,
 'eval_steps_per_second': 8.078,
 'epoch': 1.0}